# Parse Document

In [6]:
from unstructured.partition.pdf import partition_pdf
from rich import print

elements = partition_pdf(filename="../data/pdf/Riyaz_Pasha_Resume_v3.pdf", infer_table_structure=True, strategy="hi_res")

for el in elements[:10]:
    print(type(el))
    print(el.text[:200])

ImportError: cannot import name 'open_filename' from 'pdfminer.utils' (/Users/riyaz/Personal/Learning/AI/RAG/rag-tutorial-1/.venv/lib/python3.13/site-packages/pdfminer/utils.py)

In [7]:
# ============================================================
# Production-Style PDF Parsing + Semantic Chunking Pipeline
# Using Unstructured for RAG Systems
# ============================================================

# pip install:
# pip install "unstructured[pdf]" langchain rich

from pathlib import Path
from typing import List, Dict, Any

from rich import print
from rich.console import Console

from unstructured.partition.pdf import partition_pdf
from unstructured.documents.elements import (
    Title,
    NarrativeText,
    ListItem,
    Table,
)

console = Console()

# ============================================================
# CONFIG
# ============================================================

PDF_PATH = "../data/pdf/Riyaz_Pasha_Resume_v3.pdf"

# ============================================================
# STEP 1 — PARSE PDF
# ============================================================

console.rule("[bold blue]STEP 1 — Parsing PDF[/bold blue]")

elements = partition_pdf(
    filename=PDF_PATH,

    # "fast" -> lightweight
    # "hi_res" -> layout-aware parsing
    strategy="hi_res",

    # Better table extraction
    infer_table_structure=True,

    # Extract images if needed
    extract_images_in_pdf=False,

    # OCR languages
    languages=["eng"],
)

print(f"\n[green]Total Elements Extracted:[/green] {len(elements)}")

# ============================================================
# STEP 2 — INSPECT ELEMENTS
# ============================================================

console.rule("[bold blue]STEP 2 — Inspect Elements[/bold blue]")

for idx, el in enumerate(elements[:10]):

    print(f"\n[yellow]Element #{idx + 1}[/yellow]")

    print(f"[cyan]Type:[/cyan] {type(el).__name__}")

    text = el.text.strip() if el.text else ""
    print(f"[white]Text:[/white] {text[:300]}")

    if hasattr(el, "metadata"):
        print(
            f"[magenta]Page:[/magenta] "
            f"{getattr(el.metadata, 'page_number', 'N/A')}"
        )

# ============================================================
# STEP 3 — CONVERT TO STRUCTURED DOCUMENTS
# ============================================================

console.rule("[bold blue]STEP 3 — Build Structured Documents[/bold blue]")

documents: List[Dict[str, Any]] = []

for el in elements:

    text = el.text.strip() if el.text else ""

    if not text:
        continue

    doc = {
        "text": text,
        "element_type": type(el).__name__,
        "page_number": getattr(el.metadata, "page_number", None),
        "source": Path(PDF_PATH).name,
    }

    documents.append(doc)

print(f"\n[green]Structured Documents:[/green] {len(documents)}")

# ============================================================
# STEP 4 — SEMANTIC CHUNKING
# ============================================================

console.rule("[bold blue]STEP 4 — Semantic Chunking[/bold blue]")

chunks = []

current_chunk = ""
current_metadata = {}

for doc in documents:

    element_type = doc["element_type"]
    text = doc["text"]

    # --------------------------------------------------------
    # Start a new chunk whenever we encounter a title
    # --------------------------------------------------------

    if element_type == "Title":

        if current_chunk.strip():

            chunks.append({
                "content": current_chunk.strip(),
                "metadata": current_metadata,
            })

        current_chunk = text + "\n"

        current_metadata = {
            "page": doc["page_number"],
            "source": doc["source"],
            "section_title": text,
        }

    else:
        current_chunk += text + "\n"

# Add final chunk
if current_chunk.strip():

    chunks.append({
        "content": current_chunk.strip(),
        "metadata": current_metadata,
    })

print(f"\n[green]Semantic Chunks Created:[/green] {len(chunks)}")

# ============================================================
# STEP 5 — DISPLAY CHUNKS
# ============================================================

console.rule("[bold blue]STEP 5 — Preview Chunks[/bold blue]")

for idx, chunk in enumerate(chunks[:5]):

    print(f"\n[yellow]Chunk #{idx + 1}[/yellow]")

    print(f"[cyan]Metadata:[/cyan]")
    print(chunk["metadata"])

    print(f"\n[white]Content:[/white]")
    print(chunk["content"][:1000])

# ============================================================
# STEP 6 — OPTIONAL LANGCHAIN DOCUMENTS
# ============================================================

console.rule("[bold blue]STEP 6 — LangChain Documents[/bold blue]")

from langchain_core.documents import Document

langchain_docs = []

for chunk in chunks:

    doc = Document(
        page_content=chunk["content"],
        metadata=chunk["metadata"],
    )

    langchain_docs.append(doc)

print(
    f"\n[green]LangChain Documents Created:[/green] "
    f"{len(langchain_docs)}"
)

# ============================================================
# STEP 7 — READY FOR EMBEDDINGS
# ============================================================

console.rule("[bold blue]STEP 7 — Ready for Vector DB[/bold blue]")

print("""
Your pipeline is now ready for:

1. Embeddings
2. Vector Database Storage
3. Hybrid Retrieval
4. Reranking
5. RAG Pipelines

Recommended next steps:

- Qdrant
- Weaviate
- Chroma
- PGVector

and:

- BM25 Hybrid Search
- Parent-Child Retrieval
- Contextual Compression
- Rerankers
""")

ImportError: cannot import name 'open_filename' from 'pdfminer.utils' (/Users/riyaz/Personal/Learning/AI/RAG/rag-tutorial-1/.venv/lib/python3.13/site-packages/pdfminer/utils.py)

In [9]:
# ============================================================
# SIMPLE PDF READING WITH DOCLING
# ============================================================

# Install:
# pip install docling rich

from rich import print
from rich.console import Console

from docling.document_converter import DocumentConverter

console = Console()

# ============================================================
# CONFIG
# ============================================================

PDF_PATH = "../data/pdf/Riyaz_Pasha_Resume_v3.pdf"

# ============================================================
# STEP 1 — CREATE CONVERTER
# ============================================================

console.rule("[bold blue]STEP 1 — Initialize Docling[/bold blue]")

converter = DocumentConverter()

# ============================================================
# STEP 2 — CONVERT PDF
# ============================================================

console.rule("[bold blue]STEP 2 — Reading PDF[/bold blue]")

result = converter.convert(PDF_PATH)

# ============================================================
# STEP 3 — ACCESS DOCUMENT
# ============================================================

doc = result.document

print(f"\n[green]Document Loaded Successfully[/green]")

# ============================================================
# STEP 4 — EXPORT AS MARKDOWN
# ============================================================

console.rule("[bold blue]STEP 4 — Markdown Output[/bold blue]")

markdown_text = doc.export_to_markdown()

print(markdown_text[:3000])

# ============================================================
# STEP 5 — EXPORT AS JSON
# ============================================================

console.rule("[bold blue]STEP 5 — JSON Output[/bold blue]")

json_data = doc.export_to_dict()

print(json_data.keys())

# ============================================================
# STEP 6 — ITERATE THROUGH DOCUMENT ITEMS
# ============================================================

console.rule("[bold blue]STEP 6 — Document Structure[/bold blue]")

for idx, item in enumerate(doc.texts[:10]):

    print(f"\n[yellow]Item #{idx + 1}[/yellow]")

    print(f"[cyan]Label:[/cyan] {item.label}")

    print(f"[white]Text:[/white]")
    print(item.text[:500])

# ============================================================
# STEP 7 — SAVE MARKDOWN
# ============================================================

console.rule("[bold blue]STEP 7 — Save Markdown[/bold blue]")

output_path = "output.md"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(markdown_text)

print(f"\n[green]Markdown saved to:[/green] {output_path}")

/Users/riyaz/Personal/Learning/AI/RAG/rag-tutorial-1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


─────────────────────────────────────────── STEP 1 — Initialize Docling ───────────────────────────────────────────

────────────────────────────────────────────── STEP 2 — Reading PDF ───────────────────────────────────────────────

[INFO] 2026-05-25 18:02:24,654 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-25 18:02:24,659 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-25 18:02:27,018 [RapidOCR] download_file.py:82: Download size: 4.53MB
[INFO] 2026-05-25 18:02:27,773 [RapidOCR] download_file.py:95: Successfully saved to: /Users/riyaz/Personal/Learning/AI/RAG/rag-tutorial-1/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-25 18:02:27,781 [RapidOCR] main.py:57: Using /Users/riyaz/Personal/Learning/AI/RAG/rag-tutorial-1/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-25 18:02:27,833 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-25 18:02:27,834 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolv

Document Loaded Successfully

...

Label: text

Text:

Frontend:

───────────────────────────────────────────── STEP 7 — Save Markdown ──────────────────────────────────────────────

Markdown saved to: output.md